Реализовать программу для наложения фильтров на изображения.
Возможные фильтры: размытие, выделение границ, избавление от шума.
Изменить время.

Для работы с графическими файлами рекомендуется использовать libpng (man libpng). Примеры использования библиотеки в /usr/share/doc/libpng12-dev/examples/.

Считать изображение из файла, преобразовать в массив, отправить на
CUDA Device, провести на устройстве обработку фильтром, вернуть
изображение в память хоста, преобразовать в картинку, сохранить файл.
Оценить накладные расходы на организацию вычислений на GPU по сравнению
с ЦПУ.

Выделить на GPU массив arr из 10^9 элементов типа float и инициализировать его с помощью ядра следующим образом: arr[i] = sin((i%360)*Pi/180). Скопировать массив в память центрального процессора и посчитать ошибку err = sum_i(abs(sin((i%360)*Pi/180) - arr[i]))/10^9. Провести исследование зависимости результата от использования функций: sin, sinf, __sinf. Объяснить результат. Проверить результат при использовании массива типа double. Сравнить время работы этих функций с разными типами.

In [ ]:
!nvcc --version
!nvidia-smi

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0
Tue Oct 13 10:16:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8    

In [ ]:
%%writefile gpu_precision.cu
// nvcc -O2 gpu_precision.cu -o gpu_precision && ./gpu_precision
#include <cstdio>
#include <cmath>
#include <cstdlib>
#include <cuda_runtime.h>

const size_t N        = 1000000000ULL;
const double PI       = 3.14159265358979323846;
const int    THREADS  = 256;
const int    KITERS   = 256;   // sin на поток в time-ядре (compute-bound)
const int    REPS     = 10;    // повторов для усреднения
const int    WARMUP   = 3;
const double CYCLE_NS = 0.74;  // 1.35 GHz

#define CHECK(x) do{ cudaError_t e=(x); if(e!=cudaSuccess){ \
    printf("CUDA err %s:%d: %s\n",__FILE__,__LINE__,cudaGetErrorString(e)); exit(1);} }while(0)

// ---------- Error-ядра: одна sin на элемент, пишем в память ----------
__global__ void e_sin_f(float* a){
    size_t i = blockIdx.x*(size_t)blockDim.x + threadIdx.x;
    if(i < N) a[i] = sin((float)((i%360)*(float)(PI/180.0)));
}
__global__ void e_sinf(float* a){
    size_t i = blockIdx.x*(size_t)blockDim.x + threadIdx.x;
    if(i < N) a[i] = sinf((i%360)*(float)(PI/180.0));
}
__global__ void e___sinf(float* a){
    size_t i = blockIdx.x*(size_t)blockDim.x + threadIdx.x;
    if(i < N) a[i] = __sinf((i%360)*(float)(PI/180.0));
}
__global__ void e_sin_d(double* a){
    size_t i = blockIdx.x*(size_t)blockDim.x + threadIdx.x;
    if(i < N) a[i] = sin((double)(i%360)*PI/180.0);
}

// ---------- Time-ядра: KITERS sin на поток, один write ----------
template<int K>
__global__ void t_sin_f(float* a){
    size_t i = blockIdx.x*(size_t)blockDim.x + threadIdx.x;
    if(i >= N) return;
    float x = (i%360)*(float)(PI/180.0), acc = 0.f;
    #pragma unroll 8
    for(int k=0;k<K;k++){ acc += sin(x); x += 0.001f; }
    a[i] = acc;
}
template<int K>
__global__ void t_sinf(float* a){
    size_t i = blockIdx.x*(size_t)blockDim.x + threadIdx.x;
    if(i >= N) return;
    float x = (i%360)*(float)(PI/180.0), acc = 0.f;
    #pragma unroll 8
    for(int k=0;k<K;k++){ acc += sinf(x); x += 0.001f; }
    a[i] = acc;
}
template<int K>
__global__ void t___sinf(float* a){
    size_t i = blockIdx.x*(size_t)blockDim.x + threadIdx.x;
    if(i >= N) return;
    float x = (i%360)*(float)(PI/180.0), acc = 0.f;
    #pragma unroll 8
    for(int k=0;k<K;k++){ acc += __sinf(x); x += 0.001f; }
    a[i] = acc;
}
template<int K>
__global__ void t_sin_d(double* a){
    size_t i = blockIdx.x*(size_t)blockDim.x + threadIdx.x;
    if(i >= N) return;
    double x = (i%360)*PI/180.0, acc = 0.0;
    #pragma unroll 8
    for(int k=0;k<K;k++){ acc += sin(x); x += 0.001; }
    a[i] = acc;
}

// ---------- Эталон на CPU ----------
double ref_tab[360];
void buildRef(){ for(int k=0;k<360;k++) ref_tab[k]=sin(k*PI/180.0); }

// ---------- Результат одного запуска ----------
struct Res {
    const char* name;
    float  k_ms, c_ms;
    double err;
    double bw_k, bw_c;      // GB/s
    double ns_sin, cyc_sin; // на один sin
};

template<typename T>
Res run(const char* name, void(*k_err)(T*), void(*k_time)(T*), size_t bytes){
    T* d=nullptr; CHECK(cudaMalloc(&d, bytes));
    T* h=nullptr; CHECK(cudaMallocHost(&h, bytes));   // pinned host
    size_t blocks=(N+THREADS-1)/THREADS;
    cudaEvent_t t0,t1; cudaEventCreate(&t0); cudaEventCreate(&t1);

    // Ошибка
    k_err<<<blocks,THREADS>>>(d);
    CHECK(cudaDeviceSynchronize());
    CHECK(cudaMemcpy(h, d, bytes, cudaMemcpyDeviceToHost));
    double err=0.0;
    for(size_t i=0;i<N;i++) err += fabs(ref_tab[i%360] - (double)h[i]);
    err /= (double)N;

    // Прогрев
    for(int r=0;r<WARMUP;r++) k_time<<<blocks,THREADS>>>(d);
    CHECK(cudaDeviceSynchronize());

    // Kernel
    float sum_k=0;
    for(int r=0;r<REPS;r++){
        cudaEventRecord(t0);
        k_time<<<blocks,THREADS>>>(d);
        cudaEventRecord(t1); cudaEventSynchronize(t1);
        float t; cudaEventElapsedTime(&t,t0,t1); sum_k+=t;
    }
    // Copy D->H
    float sum_c=0;
    for(int r=0;r<REPS;r++){
        cudaEventRecord(t0);
        CHECK(cudaMemcpy(h,d,bytes,cudaMemcpyDeviceToHost));
        cudaEventRecord(t1); cudaEventSynchronize(t1);
        float t; cudaEventElapsedTime(&t,t0,t1); sum_c+=t;
    }

    Res R;
    R.name  = name;
    R.k_ms  = sum_k/REPS;
    R.c_ms  = sum_c/REPS;
    R.err   = err;
    R.bw_k  = bytes / (R.k_ms/1000.0) / 1e9;
    R.bw_c  = bytes / (R.c_ms/1000.0) / 1e9;
    R.ns_sin  = (double)R.k_ms * 1e6 / ((double)N * KITERS);
    R.cyc_sin = R.ns_sin / CYCLE_NS;

    cudaFreeHost(h); cudaFree(d);
    cudaEventDestroy(t0); cudaEventDestroy(t1);
    return R;
}

void printRes(const Res& r, double ref_ns_sin){
    printf("%-10s kernel=%8.2fms copy=%8.2fms err=%.6e | BW_k=%6.1f BW_c=%6.1f GB/s"
           " | %6.4f ns/sin %6.2f cyc | x%.2f\n",
           r.name, r.k_ms, r.c_ms, r.err,
           r.bw_k, r.bw_c,
           r.ns_sin, r.cyc_sin,
           ref_ns_sin / r.ns_sin);
}

int main(){
    buildRef();
    size_t bf = N*sizeof(float);
    size_t bd = N*sizeof(double);

    printf("=== FLOAT (%.0f MB, K=%d, avg of %d) ===\n", bf/1e6, KITERS, REPS);
    Res r1 = run<float>("sin(f)" , e_sin_f , t_sin_f<KITERS> , bf);
    Res r2 = run<float>("sinf"   , e_sinf  , t_sinf<KITERS>  , bf);
    Res r3 = run<float>("__sinf" , e___sinf, t___sinf<KITERS>, bf);
    double ref = r2.ns_sin;   // эталон для ускорения — sinf
    printRes(r1, ref);
    printRes(r2, ref);
    printRes(r3, ref);

    printf("=== DOUBLE (%.0f MB, K=%d, avg of %d) ===\n", bd/1e6, KITERS, REPS);
    Res r4 = run<double>("sin(d)", e_sin_d, t_sin_d<KITERS>, bd);
    printRes(r4, ref);

    return 0;
}

Overwriting gpu_precision.cu


In [ ]:
!nvcc -O2 gpu_precision.cu -o gpu_precision
!./gpu_precision

=== FLOAT (4000 MB, K=256, avg of 10) ===
sin(f)     kernel= 1465.96ms copy=  305.69ms err=4.833600e-08 | BW_k=   2.7 BW_c=  13.1 GB/s | 0.0057 ns/sin   0.01 cyc | x1.02
sinf       kernel= 1496.11ms copy=  305.32ms err=4.833600e-08 | BW_k=   2.7 BW_c=  13.1 GB/s | 0.0058 ns/sin   0.01 cyc | x1.00
__sinf     kernel=  317.60ms copy=  305.48ms err=1.409966e-07 | BW_k=  12.6 BW_c=  13.1 GB/s | 0.0012 ns/sin   0.00 cyc | x4.71
=== DOUBLE (8000 MB, K=256, avg of 10) ===
sin(d)     kernel=46134.56ms copy=  606.33ms err=8.779628e-18 | BW_k=   0.2 BW_c=  13.2 GB/s | 0.1802 ns/sin   0.24 cyc | x0.03


**Обозначения**:\
  kernel  — среднее время time-ядра (K=256 sin на поток), мс \
  copy    — среднее время cudaMemcpy D->H (pinned), мс \
  err     — средняя абс. ошибка vs CPU-эталон (double), sum|ref-h|/N \
  BW_k    — пропускная способность ядра, ГБ/с (bytes / kernel_time) \
  BW_c    — пропускная способность копирования, ГБ/с \
  ns/sin  — наносекунд на один вызов sin \
  cyc/sin — тактов на один вызов sin (CYCLE_NS=0.74 нс) \
  x       — ускорение относительно sinf (по ns/sin), >1 = быстрее

In [ ]:
%%writefile gpu_precision.cu
// nvcc -O2 gpu_precision.cu -o gpu_precision && ./gpu_precision
#include <cstdio>
#include <cmath>
#include <cstdlib>
#include <cuda_runtime.h>

const size_t N       = 1000000000ULL;   // 10^9 элементов
const double PI      = 3.14159265358979323846;
const int    THREADS = 256;
const int    K       = 256;             // sin на поток в time-ядре
const int    REPS    = 10;               // повторов для усреднения
const int    WARMUP  = 3;

#define CHECK(x) do { cudaError_t e = (x); if (e != cudaSuccess) { \
    printf("CUDA error %s:%d: %s\n", __FILE__, __LINE__, cudaGetErrorString(e)); \
    exit(1); } } while(0)

// ---------- Error-ядра: одна sin на элемент ----------
__global__ void err_sin_f(float* a) {       // полноценный double sin -> float
    size_t i = blockIdx.x * (size_t)blockDim.x + threadIdx.x;
    if (i < N) a[i] = (float)sin((double)(i % 360) * PI / 180.0);
}
__global__ void err_sinf(float* a) {        // float sinf
    size_t i = blockIdx.x * (size_t)blockDim.x + threadIdx.x;
    if (i < N) a[i] = sinf((float)(i % 360) * (float)(PI / 180.0));
}
__global__ void err___sinf(float* a) {      // быстрый intrinsic
    size_t i = blockIdx.x * (size_t)blockDim.x + threadIdx.x;
    if (i < N) a[i] = __sinf((float)(i % 360) * (float)(PI / 180.0));
}
__global__ void err_sin_d(double* a) {      // double sin
    size_t i = blockIdx.x * (size_t)blockDim.x + threadIdx.x;
    if (i < N) a[i] = sin((double)(i % 360) * PI / 180.0);
}

// ---------- Time-ядра: K sin на поток ----------
template<int K>
__global__ void time_sin_f(float* a) {
    size_t i = blockIdx.x * (size_t)blockDim.x + threadIdx.x;
    if (i >= N) return;
    double x = (i % 360) * PI / 180.0, acc = 0.0;
    #pragma unroll 8
    for (int k = 0; k < K; k++) { acc += sin(x); x += 0.001; }
    a[i] = (float)acc;
}
template<int K>
__global__ void time_sinf(float* a) {
    size_t i = blockIdx.x * (size_t)blockDim.x + threadIdx.x;
    if (i >= N) return;
    float x = (i % 360) * (float)(PI / 180.0), acc = 0.f;
    #pragma unroll 8
    for (int k = 0; k < K; k++) { acc += sinf(x); x += 0.001f; }
    a[i] = acc;
}
template<int K>
__global__ void time___sinf(float* a) {
    size_t i = blockIdx.x * (size_t)blockDim.x + threadIdx.x;
    if (i >= N) return;
    float x = (i % 360) * (float)(PI / 180.0), acc = 0.f;
    #pragma unroll 8
    for (int k = 0; k < K; k++) { acc += __sinf(x); x += 0.001f; }
    a[i] = acc;
}
template<int K>
__global__ void time_sin_d(double* a) {
    size_t i = blockIdx.x * (size_t)blockDim.x + threadIdx.x;
    if (i >= N) return;
    double x = (i % 360) * PI / 180.0, acc = 0.0;
    #pragma unroll 8
    for (int k = 0; k < K; k++) { acc += sin(x); x += 0.001; }
    a[i] = acc;
}

// ---------- Эталон на CPU ----------
double ref_tab[360];
void buildRef() { for (int k = 0; k < 360; k++) ref_tab[k] = sin(k * PI / 180.0); }

// ---------- Результат одного прогона ----------
struct Res {
    const char* name;
    double k_ms;    // время time-ядра, мс
    double c_ms;    // время cudaMemcpy D->H, мс
    double err;     // средняя абс. ошибка
    double ns_sin;  // нс на один sin
};

template<typename T>
Res run(const char* name, void(*k_err)(T*), void(*k_time)(T*), size_t bytes) {
    T* d = nullptr;
    T* h = nullptr;
    CHECK(cudaMalloc(&d, bytes));
    CHECK(cudaMallocHost(&h, bytes));            // pinned host
    size_t blocks = (N + THREADS - 1) / THREADS;

    cudaEvent_t t0, t1;
    cudaEventCreate(&t0);
    cudaEventCreate(&t1);

    // --- Ошибка ---
    k_err<<<blocks, THREADS>>>(d);
    CHECK(cudaDeviceSynchronize());
    CHECK(cudaMemcpy(h, d, bytes, cudaMemcpyDeviceToHost));
    double err = 0.0;
    for (size_t i = 0; i < N; i++) err += fabs(ref_tab[i % 360] - (double)h[i]);
    err /= (double)N;

    // --- Прогрев ---
    for (int r = 0; r < WARMUP; r++) k_time<<<blocks, THREADS>>>(d);
    CHECK(cudaDeviceSynchronize());

    // --- Время ядра ---
    double sum_k = 0.0;
    for (int r = 0; r < REPS; r++) {
        cudaEventRecord(t0);
        k_time<<<blocks, THREADS>>>(d);
        cudaEventRecord(t1);
        cudaEventSynchronize(t1);
        float t; cudaEventElapsedTime(&t, t0, t1);
        sum_k += t;
    }

    // --- Время копирования D->H ---
    double sum_c = 0.0;
    for (int r = 0; r < REPS; r++) {
        cudaEventRecord(t0);
        CHECK(cudaMemcpy(h, d, bytes, cudaMemcpyDeviceToHost));
        cudaEventRecord(t1);
        cudaEventSynchronize(t1);
        float t; cudaEventElapsedTime(&t, t0, t1);
        sum_c += t;
    }

    Res R;
    R.name   = name;
    R.k_ms   = sum_k / REPS;
    R.c_ms   = sum_c / REPS;
    R.err    = err;
    R.ns_sin = R.k_ms * 1e6 / ((double)N * K);

    cudaFreeHost(h);
    cudaFree(d);
    cudaEventDestroy(t0);
    cudaEventDestroy(t1);
    return R;
}

void printRes(const Res& r, double ref_ns_sin) {
    printf("%-10s kernel=%8.2f ms  copy=%8.2f ms  err=%.6e  |  %7.5f ns/sin  |  x%.2f\n",
           r.name, r.k_ms, r.c_ms, r.err, r.ns_sin, ref_ns_sin / r.ns_sin);
}

int main() {
    buildRef();
    size_t bf = N * sizeof(float);
    size_t bd = N * sizeof(double);

    printf("=== FLOAT (%.0f MB, K=%d, avg of %d) ===\n", bf / 1e6, K, REPS);
    Res r1 = run<float>("sin(f)",  err_sin_f,  time_sin_f<K>,  bf);
    Res r2 = run<float>("sinf",    err_sinf,   time_sinf<K>,   bf);
    Res r3 = run<float>("__sinf",  err___sinf, time___sinf<K>, bf);
    double ref = r2.ns_sin;   // эталон ускорения — sinf
    printRes(r1, ref);
    printRes(r2, ref);
    printRes(r3, ref);

    printf("=== DOUBLE (%.0f MB, K=%d, avg of %d) ===\n", bd / 1e6, K, REPS);
    Res r4 = run<double>("sin(d)", err_sin_d, time_sin_d<K>, bd);
    printRes(r4, ref);

    return 0;
}

Writing gpu_precision.cu


In [ ]:
!nvcc -O2 gpu_precision.cu -o gpu_precision
!./gpu_precision

=== FLOAT (4000 MB, K=256, avg of 10) ===
sin(f)     kernel=46448.72 ms  copy=  304.98 ms  err=1.198678e-08  |  0.18144 ns/sin  |  x0.04
sinf       kernel= 1627.64 ms  copy=  305.41 ms  err=4.833600e-08  |  0.00636 ns/sin  |  x1.00
__sinf     kernel=  337.52 ms  copy=  309.36 ms  err=1.409966e-07  |  0.00132 ns/sin  |  x4.82
=== DOUBLE (8000 MB, K=256, avg of 10) ===
sin(d)     kernel=46437.51 ms  copy=  610.74 ms  err=8.779628e-18  |  0.18140 ns/sin  |  x0.04
